In [1]:
import numpy as np
import os
import pandas as pd
from sklearn.cluster import DBSCAN

In [2]:
TRAIN_LOCS_KEY = 'train_locs'
TRAIN_IDS_KEY = 'train_ids'
TAXON_IDS_KEY = 'taxon_ids'
TAXON_NAME_KEY = 'taxon_names'

Reading the file:

In [3]:
filepath = os.path.join(os.getcwd(), 'species_train.npz')
data = np.load(filepath, allow_pickle=True)
train_locs = data[TRAIN_LOCS_KEY]
train_ids = data[TRAIN_IDS_KEY]
taxon_ids = data[TAXON_IDS_KEY]
taxon_names = data[TAXON_NAME_KEY]

Mapping the taxon ids to taxon latin names: 

In [4]:
species_ids_names = dict(zip(data['taxon_ids'], data['taxon_names']))  # latin names of species 

Create pandas Dataframe from the data: 

In [5]:
df = pd.DataFrame({
    'latitude': train_locs[:, 0],
    'longitude': train_locs[:, 1], 
    'taxon_id': data['train_ids']
})
df['taxon_name'] = [species_ids_names[id] for id in data['train_ids'].astype(int)]
df.head()

,latitude,longitude,taxon_id,taxon_name
0,-18.286728,143.481247,31529,Lophognathus gilberti
1,-13.099798,130.783646,31529,Lophognathus gilberti
2,-13.965274,131.695145,31529,Lophognathus gilberti
3,-12.853950,132.800507,31529,Lophognathus gilberti
4,-12.196790,134.279327,31529,Lophognathus gilberti


In [6]:
df.shape

(272037, 4)

Data Cleanining: 

<small>1. Check for missing or invalid coordinates:</small>

In [7]:
df = df.dropna(subset=['latitude', 'longitude'])
df = df[(df['latitude'].between(-90, 90)) & (df['longitude'].between(-180, 180))]
df.shape

(272037, 4)

<small>2. Remove any duplicates or nearly duplicates (observations that are extremely close):</small>

In [8]:
df['lat_rounded'] = df['latitude'].round(5)
df['lon_rounded'] = df['longitude'].round(5)

In [12]:
df = df.drop_duplicates(subset=['lat_rounded', 'lon_rounded'])
df.shape

(237977, 6)

<small>
<div>In order to remove nearly duplicate, DBSCAN is used:</div>
<div>DBSCAN: Density-Based Spatial Clustering of Applications with Noise</div>
<div>This is an algorithm that finds neighbours in a set of data.</div>
<div>Parameters Choosing:</div>
<ol>
<li>epsilon was chosen = 10 meters, so any 2 points with space less than or equal to 10 meters are comsidered neighbours.</li>
<li>min_value was chosen = 1, so any cluster will consist of 1 or more items. This is because we are interested in deduplication, not in finding dense clusters.</li>
<li>the metric that was used is haversine, because we are computing distances on a sphere.</li>
<li>the algorithm that was used is ball_tree, because it's the efficient spatial lookup for haversine metric.</li>
</ol>
</small>

In [ ]:
# # remove nearly duplicates
# # 1. convert to radians for haversine metric
# coords = np.radians(df[['latitude', 'longitude']])  
# # 2. Define earth's radius
# earth_radius = 6371.0088
# # 3. define epsilon: 
# eps = 0.01/earth_radius
# # 4. fit the DBSCAN
# db = DBSCAN(
#     eps=eps,
#     min_samples=1,
#     metric='haversine',
#     algorithm='ball_tree'
# ).fit(coords)
# #5. check if there are neighbours
# df['cluster_id'] = db.labels_

In [ ]:
# df = df.groupby(['cluster_id', 'taxon_id']).first().reset_index(drop=True)

In [13]:
df.head()

,latitude,longitude,taxon_name
0,-18.286728,143.481247,Pachycephala rufiventris
1,-18.286728,143.481247,Lichmera indistincta
2,-18.286728,143.481247,Pardalotus striatus
3,-18.286728,143.481247,Pomatostomus temporalis
4,-18.286728,143.481247,Lophognathus gilberti


<small>4. Validate species IDs: </small>

<small>5. Check class imbalance</small>

<small>6. Check spatial outliers</small>

<small>7. Convert to categorical labels:</small>

<small>5. Only keep birds:</small>

<small>6. Split the data to x and y</small>

<small>7. Split to train and validation sets</small>